# WildChat-4.8M-Full vs Dolci Instruct: which WildChat conversations reached post-training?

**Goal (mentor task 1, first step):** build a slim index of every WildChat conversation (toxic and non-toxic) and
check how the Dolci Instruct post-training data links back to it.

**Data:**
- `yuntian-deng/WildChat-4.8M-Full` (gated; the copy your Hugging Face account is approved for). 127 parquet files,
  21.4 GB. We **stream** it and keep only small columns, so the full dataset never touches the disk.
- `allenai/Dolci-Instruct-RL`, which carries WildChat's `conversation_hash` on its WildChat-sourced rows, so the join
  is exact. (Dolci SFT and DPO have no hash; matching them needs prompt text, see section 5.)

**Caveats:**
- Some Dolci source names point to *WildChat-1M* (2023-04 to 2024-04), but the RL rows reach into 2025, so Dolci drew
  on later WildChat data too. 4.8M-Full runs to 2025-07, so section 4 cuts the denominator at Dolci's last date.
- `conversation_hash` is a hash of the conversation content, so identical conversations share one (one hash appears
  24,688 times). The toxic flag is always the same within a hash, so we count **unique hashes** as conversations.

## 0. Setup

Run from the project root, e.g.
```bash
uv run --no-project --with jupyterlab,pandas,pyarrow,huggingface_hub jupyter lab
```
You need to be logged in to Hugging Face (`hf auth login`) with the account that has WildChat-Full access.

In [1]:
import io
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")  # default 10s is too short for ~110 MB files

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfFileSystem, snapshot_download
from IPython.display import display

# Locate the project root (works whether the notebook is run from the root or from notebooks/).
ROOT = Path.cwd()
while not (ROOT / "helper_scripts" / "core_rates.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from inside the project folder.")
    ROOT = ROOT.parent

# Pinned revisions so the numbers are reproducible.
WILDCHAT = ("yuntian-deng/WildChat-4.8M-Full", "462d2dc4d4189436bc0d54313b4fd925f1fe93eb")
DOLCI_RL = ("allenai/Dolci-Instruct-RL", "1a8d79b1012d77e4bb1fd7f699c5c7eb33ef931e")

# Local cache for the slim index. `data/` is gitignored.
CACHE = ROOT / "data" / "wildchat"
CACHE.mkdir(parents=True, exist_ok=True)
INDEX = CACHE / "wildchat48m_full_index.parquet"

# Cheap columns only: `conversation` (the text) is the bulk of the 21 GB and isn't needed for this step.
INDEX_COLS = ["conversation_hash", "model", "timestamp", "turn", "language", "toxic", "redacted"]

pd.set_option("display.width", 200)

/Users/mihirjoshi/.cache/uv/archive-v0/M7PJ5G2dkOVLU2RN/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Build (or load) the slim WildChat index

The first run streams all 127 files over the network, 4 at a time (roughly 15-25 minutes, depending on your
connection), and saves only the index columns to `data/wildchat/`. Later runs load the local file in seconds.

In [2]:
# Each ~110 MB file is fetched whole into memory in one request (reading only some columns straight from the
# server takes hundreds of tiny requests and is ~3x slower), the slim columns are kept, and the bytes are dropped.
# Nothing big is written to disk. Each file's slim part is saved to parts/, so an interrupted run resumes.
PARTS = CACHE / "parts"
PARTS.mkdir(exist_ok=True)


def fetch_slim(path, attempts=5):
    out = PARTS / Path(path).name
    if out.exists():
        return out
    for attempt in range(1, attempts + 1):
        try:
            buf = io.BytesIO(fs.cat_file(path))
            break
        except Exception as e:  # network hiccups (e.g. ReadTimeout): wait and retry
            if attempt == attempts:
                raise
            print(f"  retry {attempt} for {Path(path).name}: {type(e).__name__}", flush=True)
            time.sleep(10 * attempt)
    pq.write_table(pq.read_table(buf, columns=INDEX_COLS), out)
    return out


if INDEX.exists():
    wc = pd.read_parquet(INDEX)
    print(f"Loaded cached index: {INDEX.relative_to(ROOT)}")
else:
    fs = HfFileSystem()
    repo, rev = WILDCHAT
    files = sorted(fs.glob(f"datasets/{repo}@{rev}/data/*.parquet"))
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, _ in enumerate(pool.map(fetch_slim, files), 1):
            if i % 10 == 0 or i == len(files):
                print(f"  {i}/{len(files)} files, {time.time() - t0:.0f}s", flush=True)
    table = pa.concat_tables(pq.read_table(PARTS / Path(p).name) for p in files)
    pq.write_table(table, INDEX)
    wc = table.to_pandas()
    print(f"Saved {INDEX.relative_to(ROOT)} ({INDEX.stat().st_size / 1e6:.0f} MB)")

print(f"{len(wc):,} conversations, {wc.conversation_hash.nunique():,} unique hashes")

Loaded cached index: data/wildchat/wildchat48m_full_index.parquet


4,743,336 conversations, 4,292,113 unique hashes


## 2. How much of WildChat is flagged toxic?

`toxic` is WildChat's own flag (OpenAI moderation or Detoxify fired on any turn). The public releases drop exactly
these rows.

In [3]:
uniq = wc.drop_duplicates("conversation_hash")  # one row per distinct conversation
summary = pd.DataFrame({
    "rows": [len(wc), len(uniq)],
    "toxic": [int(wc.toxic.sum()), int(uniq.toxic.sum())],
    "toxic %": [round(100 * wc.toxic.mean(), 1), round(100 * uniq.toxic.mean(), 1)],
    "non-toxic": [int((~wc.toxic).sum()), int((~uniq.toxic).sum())],
}, index=["all rows (non-toxic rows = public 4.8M)", "unique conversations"])
display(summary)

by_lang = (wc.groupby("language").toxic.agg(conversations="size", toxic="sum")
             .sort_values("conversations", ascending=False).head(10))
by_lang["toxic %"] = (100 * by_lang.toxic / by_lang.conversations).round(1)
display(by_lang)

,rows,toxic,toxic %,non-toxic
all rows (non-toxic rows = public 4.8M),4743336,1543476,32.5,3199860
unique conversations,4292113,1423031,33.2,2869082


,conversations,toxic,toxic %
language,,,
English,2297065,617694,26.9
Russian,1160333,797068,68.7
Chinese,276523,10265,3.7
French,182450,9283,5.1
Vietnamese,129765,2267,1.7
Yoruba,99280,7446,7.5
Indonesian,84798,17761,20.9
Arabic,81691,4204,5.1
Spanish,81043,20722,25.6


## 3. Exact join: Dolci Instruct RL → WildChat

Dolci RL rows from `hamishivi/new-wildchat-english-general_filtered` keep the original `conversation_hash`.

In [4]:
rl_dir = Path(snapshot_download(DOLCI_RL[0], repo_type="dataset", revision=DOLCI_RL[1],
                                allow_patterns=["data/*.parquet"]))
rl = pd.concat(pd.read_parquet(f, columns=["conversation_hash"]) for f in sorted(rl_dir.glob("data/*.parquet")))
rl_hashes = set(rl.conversation_hash.dropna())

wc["in_dolci_rl"] = wc.conversation_hash.isin(rl_hashes)
found = wc[wc.in_dolci_rl]

join = pd.DataFrame({
    "Dolci RL rows with a WildChat hash": [len(rl_hashes)],
    "found in WildChat-4.8M-Full": [found.conversation_hash.nunique()],
    "not found": [len(rl_hashes - set(found.conversation_hash))],
    "found AND flagged toxic": [int(found.toxic.sum())],
})
display(join)

# For comparison: the public (non-toxic) WildChat-1M matched 8,397 of these hashes (checked 2026-11-06); the rest are
# conversations logged after WildChat-1M's cutoff, not toxic ones.
toxic_rate = pd.DataFrame({
    "toxic % in Dolci RL's WildChat rows": [round(100 * found.toxic.mean(), 2)],
    "toxic % in all WildChat": [round(100 * wc.toxic.mean(), 2)],
})
display(toxic_rate)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 1468.59it/s]

,Dolci RL rows with a WildChat hash,found in WildChat-4.8M-Full,not found,found AND flagged toxic
0,10621,10621,0,0


,toxic % in Dolci RL's WildChat rows,toxic % in all WildChat
0,0.0,32.54


## 4. Restrict to the time window Dolci could have drawn from

Conversations logged after Dolci's last WildChat conversation can't have been selected, so the fairer denominator for
"of all toxic conversations, how many made it through" stops at that date. Counts are unique conversations.

In [5]:
ts = pd.to_datetime(wc.timestamp, utc=True)
print("4.8M-Full spans", ts.min().date(), "to", ts.max().date())
print("Dolci RL matches span", ts[wc.in_dolci_rl].min().date(), "to", ts[wc.in_dolci_rl].max().date())

window = wc[ts <= ts[wc.in_dolci_rl].max()].drop_duplicates("conversation_hash")
display(pd.DataFrame({
    "conversations in window": [len(window)],
    "toxic in window": [int(window.toxic.sum())],
    "toxic that reached Dolci RL": [int((window.toxic & window.in_dolci_rl).sum())],
    "non-toxic that reached Dolci RL": [int((~window.toxic & window.in_dolci_rl).sum())],
}))

4.8M-Full spans 2023-04-09 to 2025-07-31
Dolci RL matches span 2023-04-10 to 2025-02-10


,conversations in window,toxic in window,toxic that reached Dolci RL,non-toxic that reached Dolci RL
0,3280552,1141725,0,10621


## 5. Story filter: WildChat-sourced stories in Dolci Instruct

"Story" = the paper's own story extracts (`training_stories/data/`), so the set matches Table 2. Those extracts were
cut from AllenAI's pre-release datasets, so their `id`s are row numbers there and don't join to released Dolci,
except DPO, whose `prompt_id` does. Each stage joins differently:

| Stage | Join to released Dolci | WildChat rows |
|---|---|---|
| SFT | extract `text` = the row's assistant response | `source_dataset == "Wildchat"` |
| DPO | extract `id` = `prompt_id` | `prompt_id` starts with a WildChat prefix |
| RL | extract `text` = one of the row's `ground_truth` reference answers | `conversation_hash` is set |

The README warns that the extracts were selected automatically and include non-stories, so `story_request` adds a
stricter check on the prompt itself: does the user actually ask for fiction?

In [6]:
DOLCI_SFT = ("allenai/Dolci-Instruct-SFT", "bd3c8f3a9b2cc5a9682e44b96ddd0bb2ff027221")
DOLCI_DPO = ("allenai/Dolci-Instruct-DPO", "aed155cf32e809b590490b6c3577ee4b0d0a5019")
EXTRACTS = ROOT / "training_stories" / "data"
WILDCHAT_DPO_PREFIXES = ("filtered_wc_sample_500k", "Wildchat-1M-gpt-4.1-regenerated-english",
                         "Wildchat-1m-gpt-4.1-regeneration-not-english")

# A prompt counts as a story request if it names a kind of fiction, or is written as a script for the model to
# continue ("(In the clubroom...)" stage directions, or `Name: "..."` dialogue lines, common in WildChat roleplay).
# Deliberately simple, so it is easy to audit; check the misses printed below.
STORY_WORDS = re.compile(
    r"\b(?:stor(?:y|ies)|tale|fable|fanfic\w*|fan[- ]fiction|narrative|novel|chapter|plot|script|screenplay|"
    r"episode|scene|scenario|dialog(?:ue)?|backstory|crossover|roleplay|role-play|fiction|bedtime|legend|myth|saga)\b",
    re.I)
SCRIPT_FORMAT = re.compile(r"^\s*[(\[]|\b[A-Z][\w.\-]*(?: [A-Z][\w.\-]*)?: [\"“]")


def is_story_request(prompt):
    return bool(STORY_WORDS.search(prompt) or SCRIPT_FORMAT.search(prompt))


def norm(text):
    return " ".join(str(text).split())


def read_dolci(repo_rev, columns, filters=None):
    path = Path(snapshot_download(repo_rev[0], repo_type="dataset", revision=repo_rev[1],
                                  allow_patterns=["data/*.parquet"]))
    return pd.concat([pd.read_parquet(f, columns=columns, filters=filters) for f in sorted(path.glob("data/*.parquet"))],
                     ignore_index=True)


def extract(name):
    return pd.read_json(EXTRACTS / f"{name}.jsonl", lines=True, dtype={"id": str})


# SFT: WildChat rows whose response is one of the paper's story texts.
sft_texts = set(extract("dolci-instruct-sft").text.map(norm))
sft = read_dolci(DOLCI_SFT, ["id", "messages"], filters=[("source_dataset", "==", "Wildchat")])
sft = sft[sft.messages.map(lambda m: norm(m[-1]["content"]) in sft_texts)]
sft_st = pd.DataFrame({"stage": "sft", "key": "sft:" + sft.id, "prompt": sft.messages.map(lambda m: m[0]["content"])})

# DPO: the extract ids are prompt_ids; keep the WildChat-sourced ones.
dpo_ids = set(pd.read_csv(EXTRACTS / "dolci_record_ids.csv", dtype=str).query("dolci_file == 'dolci-instruct-dpo'").id)
dpo = read_dolci(DOLCI_DPO, ["prompt_id", "chosen"])
dpo = dpo[dpo.prompt_id.isin(dpo_ids) & dpo.prompt_id.str.startswith(WILDCHAT_DPO_PREFIXES)]
dpo = dpo.drop_duplicates("prompt_id")
dpo_st = pd.DataFrame({"stage": "dpo", "key": "dpo:" + dpo.prompt_id, "prompt": dpo.chosen.map(lambda m: m[0]["content"])})

# RL: rows whose reference answer is one of the paper's story texts and that carry a WildChat hash.
rl_texts = set(extract("dolci-instruct-rl").text.map(norm))
rl_all = read_dolci(DOLCI_RL, ["id", "prompt", "ground_truth", "conversation_hash"])
rl_hit = rl_all.ground_truth.map(lambda g: g is not None and any(norm(x) in rl_texts for x in g))
rl = rl_all[rl_hit & rl_all.conversation_hash.notna()]
rl_st = pd.DataFrame({"stage": "rl", "key": "rl:" + rl.id, "conversation_hash": rl.conversation_hash,
                      # RL prompts flatten the conversation ("user: ...\nassistant: ..."); keep the first user turn
                      "prompt": rl.prompt.str.removeprefix("user: ").str.split("\nassistant:").str[0]})

stories = pd.concat([sft_st, dpo_st, rl_st], ignore_index=True)
stories["story_request"] = stories.prompt.map(is_story_request)

# WildChat's own flag: exact for RL (hash join); SFT/DPO need prompt matching first (not done yet).
flags = wc.drop_duplicates("conversation_hash").set_index("conversation_hash").toxic
stories["wc_toxic"] = stories.conversation_hash.map(flags)

STAGES = ["sft", "dpo", "rl"]
tab = stories.groupby("stage").agg(stories=("key", "size"), story_request=("story_request", "sum"),
                                   wc_flag_known=("wc_toxic", "count")).reindex(STAGES)
tab["story_request %"] = (100 * tab.story_request / tab.stories).round(1)
display(tab)

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Fetching 15 files: 100%|██████████| 15/15 [00:00<00:00, 5119.58it/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 2711.69it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 2429.13it/s]

,stories,story_request,wc_flag_known,story_request %
stage,,,,
sft,43123,36962,0,85.7
dpo,1810,1405,0,77.6
rl,844,581,844,68.8


Spot-check the filter: prompts the paper counted as stories that **fail** the story-request check. Read these
before trusting either definition (prompts are cut to 160 characters; some WildChat content is explicit).

In [7]:
for stage in STAGES:
    miss = stories[(stories.stage == stage) & ~stories.story_request]
    print(f"--- {stage}: {len(miss):,} without a story word; 5 random examples")
    for p in miss.sample(min(5, len(miss)), random_state=0).prompt:
        print("  ", norm(p)[:160])

--- sft: 6,161 without a story word; 5 random examples
   write a sroy based on the follwing i have a converation with pokimane during the conversation as she chuckles another sound can be heard comming form her.
   Write a full length battle between Peppino and Zamasu Peppino Abilities ------------------- Jump: Peppino can jump on top of enemies, stunning them and making t
   Freedom planet react to Naruto vs Tiering System. Naruto vs Wall. Naruto win. Naruto vs building. Naruto win. Naruto vs Town. Naruto win. Naruto vs City. Naruto
   Write a cuckquean joke of 500 words
   Make a Part 1, Title: When Worlds Collide - Part 1, about What if Sonic (AOSTH), SpongeBob (1-3 Seasons), Patrick Star (1-3 Seasons), and Tails (AOSTH) as Littl
--- dpo: 405 without a story word; 5 random examples
   write a akira tozawa vs ultimo dragon with tozawa win
   write an email to CPF Singapore complaining about the money still have not deposit into bank account although the account has been closed for 1

Save the story table, plus the classifier's input: every DPO and RL story and a seeded random 5,000 of the
43K SFT stories (enough for about ±1 percentage point on a rate; rerun with all of them if you need more).

In [8]:
SFT_SAMPLE = 5_000
STORIES = CACHE / "dolci_instruct_wildchat_stories.parquet"
GUARD_IN = CACHE / "guard_input.jsonl"
GUARD_OUT = CACHE / "guard_labels.jsonl"

stories.to_parquet(STORIES)
guard_in = pd.concat([stories[stories.stage == "sft"].sample(SFT_SAMPLE, random_state=0),
                      stories[stories.stage != "sft"]])
guard_in[["key", "prompt"]].to_json(GUARD_IN, orient="records", lines=True, force_ascii=False)
print(f"{len(stories):,} stories -> {STORIES.relative_to(ROOT)}")
print(f"{len(guard_in):,} prompts to label -> {GUARD_IN.relative_to(ROOT)}")

45,777 stories -> data/wildchat/dolci_instruct_wildchat_stories.parquet
7,654 prompts to label -> data/wildchat/guard_input.jsonl


## 6. A newer classifier vs WildChat's flags

WildChat's `toxic` flag comes from the 2023 OpenAI moderation endpoint and Detoxify. We relabel each story **prompt**
with **Qwen3Guard-Gen-4B** (Qwen, 2025), a multilingual safety classifier that answers *Safe / Controversial /
Unsafe* and names categories (sexual content, violence, self-harm, ...). It runs locally with llama.cpp:

```bash
~/src/llama.cpp/build/bin/llama-server -m ~/models/gguf/Qwen3Guard-Gen-4B-Q8_0.gguf --jinja \
    -c 32768 --parallel 4 --port 8081
uv run --no-project helper_scripts/guard_classify.py data/wildchat/guard_input.jsonl \
    --out data/wildchat/guard_labels.jsonl --workers 4
```

Every story here reached Dolci, and the RL ones are confirmed non-toxic by WildChat (section 3), so anything the
classifier calls Unsafe is toxicity WildChat's flags missed.

In [9]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    centre, half = (p + z * z / (2 * n)) / (1 + z * z / n), z * ((p * (1 - p) + z * z / (4 * n)) / n) ** 0.5 / (1 + z * z / n)
    return 100 * (centre - half), 100 * (centre + half)


if not GUARD_OUT.exists():
    print(f"No labels yet: run the two commands above, then rerun this section.")
else:
    labels = pd.read_json(GUARD_OUT, lines=True)
    lab = guard_in.merge(labels[["key", "safety", "categories", "truncated"]], on="key")
    print(f"{len(lab):,} of {len(guard_in):,} prompts labelled; unparsed replies: {lab.safety.isna().sum()}")

    def rates(df):
        rows = {}
        for stage in STAGES:
            d = df[df.stage == stage]
            for label in ["Unsafe", "Controversial"]:
                k = int((d.safety == label).sum())
                lo, hi = wilson(k, len(d))
                rows[(stage, label)] = {"n": len(d), "count": k, "%": round(100 * k / max(len(d), 1), 1),
                                        "95% CI": f"[{lo:.1f}, {hi:.1f}]"}
        return pd.DataFrame.from_dict(rows, orient="index")

    print("All story prompts that reached Dolci:")
    display(rates(lab))
    print("Only prompts that pass the story-request check:")
    display(rates(lab[lab.story_request]))

    print("WildChat flag vs Qwen3Guard (RL, where the flag is known exactly):")
    display(pd.crosstab(lab.wc_toxic.dropna().map({True: "WildChat toxic", False: "WildChat non-toxic"}),
                        lab.loc[lab.wc_toxic.notna(), "safety"]))

    cats = lab[lab.safety == "Unsafe"].explode("categories")
    display(cats.groupby(["categories", "stage"]).size().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0)
                .sort_values("sft", ascending=False))

No labels yet: run the two commands above, then rerun this section.


Read what the classifier flags before reporting a number (cut to 200 characters; explicit content likely).

In [10]:
if GUARD_OUT.exists():
    for stage in STAGES:
        d = lab[(lab.stage == stage) & (lab.safety == "Unsafe")]
        print(f"--- {stage}: {len(d):,} Unsafe; 5 random examples")
        for _, r in d.sample(min(5, len(d)), random_state=0).iterrows():
            print(f"   [{', '.join(r.categories)}]", norm(r.prompt)[:200])

## 7. Next steps

- **WildChat flags for SFT / DPO.** These stages have no `conversation_hash`, so the flag column is empty for them.
  Matching needs WildChat's first user messages (the `conversation` column) plus exact-then-fuzzy prompt matching.
  The RL rows are ground truth for testing the matcher.
- **Stories on the WildChat side.** The same `STORY_REQUEST` check, applied to WildChat's first user messages, gives
  the denominator for "of all toxic *stories* in WildChat, how many reached Dolci".
- **Responses.** Qwen3Guard can also judge a prompt + response pair; Dolci's responses were rewritten by teacher
  models, so labelling them shows whether the stories themselves are unsafe, not just the requests.